In [1]:
%pip install pandas scikit-learn matplotlib nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
DOCS_DIR = PROJECT_DIR / "documents"
OUTPUT_DIR = PROJECT_DIR / "outputs"

DOCS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

documents = {
    "python.txt": """
Python is a high-level programming language.
Python is used in automation, web development,
data analysis, artificial intelligence and machine learning.
Pandas is a Python library used for data manipulation.
Scikit-learn provides tools for machine learning.
""",

    "machine_learning.txt": """
Machine learning is a branch of artificial intelligence.
It enables computers to learn patterns from data.
Supervised learning uses labelled training examples.
Classification predicts categories.
Regression predicts numerical values.
""",

    "college.txt": """
Students should carry their college identification cards.
Students should follow college attendance rules.
Laboratory sessions provide practical learning experience.
Students should submit assignments before deadlines.
""",

    "project.txt": """
A document question-answering assistant answers questions
using information from a collection of documents.
The system splits documents into smaller text chunks.
TF-IDF represents text using term importance.
Cosine similarity compares questions with document passages.
The assistant displays supporting passages and source filenames.
If information is unavailable, it reports that it was not found.
"""
}

for filename, content in documents.items():
    (DOCS_DIR / filename).write_text(
        content.strip(),
        encoding="utf-8"
    )

print("Sample documents created successfully!")

for file in sorted(DOCS_DIR.glob("*.txt")):
    print(file.name)

Sample documents created successfully!
college.txt
machine_learning.txt
project.txt
python.txt


In [3]:
corpus = {}

for file in sorted(DOCS_DIR.glob("*.txt")):
    text = file.read_text(encoding="utf-8").strip()

    if text:
        corpus[file.name] = text

print("Total documents:", len(corpus))

for filename, text in corpus.items():
    print(filename, "-", len(text.split()), "words")

Total documents: 4
college.txt - 25 words
machine_learning.txt - 29 words
project.txt - 53 words
python.txt - 35 words


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

chunk_texts = [chunk["text"] for chunk in chunks]

tfidf_matrix = vectorizer.fit_transform(chunk_texts)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Document retrieval system is ready!")

TF-IDF matrix shape: (4, 183)
Document retrieval system is ready!


In [13]:
import re
import pandas as pd

def create_chunks(text, max_words=80, overlap_sentences=1):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    current = []

    for sentence in sentences:
        if not sentence.strip():
            continue

        if current and len((" ".join(current + [sentence])).split()) > max_words:
            chunks.append(" ".join(current))
            current = current[-overlap_sentences:] if overlap_sentences else []

        current.append(sentence)

    if current:
        chunks.append(" ".join(current))

    return chunks


chunks = []

for filename, text in corpus.items():
    for i, chunk_text in enumerate(create_chunks(text)):
        chunks.append({
            "source": filename,
            "chunk_id": i + 1,
            "text": chunk_text
        })

chunks_df = pd.DataFrame(chunks)

print("Total text chunks:", len(chunks_df))
display(chunks_df)


Total text chunks: 4


,source,chunk_id,text
0,college.txt,1,Students should carry their college identifica...
1,machine_learning.txt,1,Machine learning is a branch of artificial int...
2,project.txt,1,A document question-answering assistant answer...
3,python.txt,1,Python is a high-level programming language. P...


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunk_texts = [chunk["text"] for chunk in chunks]

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(chunk_texts)

print("Chunks:", len(chunks))
print("Vocabulary size:", len(vectorizer.vocabulary_))
print("TF-IDF system rebuilt!")

Chunks: 4
Vocabulary size: 183
TF-IDF system rebuilt!


In [6]:
import re

MIN_SIMILARITY = 0.12

def answer_question(question, top_k=3):
    question = question.strip()

    if not question:
        return {"answer": "Please enter a question."}

    query_vector = vectorizer.transform([question])

    scores = cosine_similarity(
        query_vector, tfidf_matrix
    ).flatten()

    ranked_indices = scores.argsort()[::-1][:top_k]
    best_idx = int(ranked_indices[0])
    best_score = float(scores[best_idx])

    retrieved = []

    for idx in ranked_indices:
        retrieved.append({
            "source": chunks[idx]["source"],
            "chunk_id": chunks[idx]["chunk_id"],
            "passage": chunks[idx]["text"],
            "score": round(float(scores[idx]), 3)
        })

    if best_score < MIN_SIMILARITY:
        return {
            "answer": "Information not found in provided document.",
            "source": None,
            "passage": None,
            "score": best_score,
            "retrieved": pd.DataFrame(retrieved)
        }

    best_chunk = chunks[best_idx]

    sentences = re.split(
        r"(?<=[.!?])\s+",
        best_chunk["text"]
    )

    sentence_vectors = vectorizer.transform(sentences)

    sentence_scores = cosine_similarity(
        query_vector, sentence_vectors
    ).flatten()

    best_sentence_idx = int(sentence_scores.argmax())

    return {
        "answer": sentences[best_sentence_idx],
        "source": best_chunk["source"],
        "passage": best_chunk["text"],
        "score": best_score,
        "retrieved": pd.DataFrame(retrieved)
    }


def display_answer(question):
    result = answer_question(question)

    print("Question:", question)
    print("\nAnswer:", result["answer"])
    print("Similarity score:", round(result.get("score", 0), 3))

    if result.get("source"):
        print("\nSource document:", result["source"])
        print("Supporting passage:", result["passage"])
    else:
        print("\nNo sufficiently relevant source passage found.")

    if "retrieved" in result:
        print("\nTop retrieved passages:")
        display(result["retrieved"])

    return result

In [7]:
display_answer("What is machine learning?")

Question: What is machine learning?

Answer: Scikit-learn provides tools for machine learning.
Similarity score: 0.325

Source document: python.txt
Supporting passage: Python is a high-level programming language. Python is used in automation, web development, data analysis, artificial intelligence and machine learning. Pandas is a Python library used for data manipulation. Scikit-learn provides tools for machine learning.

Top retrieved passages:


,source,chunk_id,passage,score
0,python.txt,1,Python is a high-level programming language. P...,0.325
1,machine_learning.txt,1,Machine learning is a branch of artificial int...,0.242
2,college.txt,1,Students should carry their college identifica...,0.047


{'answer': 'Scikit-learn provides tools for machine learning.',
 'source': 'python.txt',
 'passage': 'Python is a high-level programming language. Python is used in automation, web development, data analysis, artificial intelligence and machine learning. Pandas is a Python library used for data manipulation. Scikit-learn provides tools for machine learning.',
 'score': 0.324825032002327,
 'retrieved':                  source  chunk_id  \
 0            python.txt         1   
 1  machine_learning.txt         1   
 2           college.txt         1   
 
                                              passage  score  
 0  Python is a high-level programming language. P...  0.325  
 1  Machine learning is a branch of artificial int...  0.242  
 2  Students should carry their college identifica...  0.047  }

In [8]:
display_answer(
    "Which Python library is used for data manipulation?"
)

Question: Which Python library is used for data manipulation?

Answer: Pandas is a Python library used for data manipulation.
Similarity score: 0.527

Source document: python.txt
Supporting passage: Python is a high-level programming language. Python is used in automation, web development, data analysis, artificial intelligence and machine learning. Pandas is a Python library used for data manipulation. Scikit-learn provides tools for machine learning.

Top retrieved passages:


,source,chunk_id,passage,score
0,python.txt,1,Python is a high-level programming language. P...,0.527
1,machine_learning.txt,1,Machine learning is a branch of artificial int...,0.032
2,project.txt,1,A document question-answering assistant answer...,0.000


{'answer': 'Pandas is a Python library used for data manipulation.',
 'source': 'python.txt',
 'passage': 'Python is a high-level programming language. Python is used in automation, web development, data analysis, artificial intelligence and machine learning. Pandas is a Python library used for data manipulation. Scikit-learn provides tools for machine learning.',
 'score': 0.5271082501881932,
 'retrieved':                  source  chunk_id  \
 0            python.txt         1   
 1  machine_learning.txt         1   
 2           project.txt         1   
 
                                              passage  score  
 0  Python is a high-level programming language. P...  0.527  
 1  Machine learning is a branch of artificial int...  0.032  
 2  A document question-answering assistant answer...  0.000  }

In [9]:
display_answer(
    "Who won the FIFA World Cup in 2034?"
)

Question: Who won the FIFA World Cup in 2034?

Answer: Information not found in provided document.
Similarity score: 0.0

No sufficiently relevant source passage found.

Top retrieved passages:


,source,chunk_id,passage,score
0,python.txt,1,Python is a high-level programming language. P...,0.0
1,project.txt,1,A document question-answering assistant answer...,0.0
2,machine_learning.txt,1,Machine learning is a branch of artificial int...,0.0


{'answer': 'Information not found in provided document.',
 'source': None,
 'passage': None,
 'score': 0.0,
 'retrieved':                  source  chunk_id  \
 0            python.txt         1   
 1           project.txt         1   
 2  machine_learning.txt         1   
 
                                              passage  score  
 0  Python is a high-level programming language. P...    0.0  
 1  A document question-answering assistant answer...    0.0  
 2  Machine learning is a branch of artificial int...    0.0  }

In [10]:
test_questions = [
    "What is Python used for?",
    "What is supervised learning?",
    "Which library is used for data manipulation?",
    "What does cosine similarity compare?",
    "What are the college attendance rules?",
    "Who won the FIFA World Cup in 2034?"
]

test_results = []

for question in test_questions:
    result = answer_question(question)

    test_results.append({
        "Question": question,
        "Answer": result["answer"],
        "Source": result.get("source") or "Not found",
        "Similarity": round(result.get("score", 0), 3)
    })

test_df = pd.DataFrame(test_results)

display(test_df)

,Question,Answer,Source,Similarity
0,What is Python used for?,"Python is used in automation, web development,...",python.txt,0.438
1,What is supervised learning?,Supervised learning uses labelled training exa...,machine_learning.txt,0.274
2,Which library is used for data manipulation?,Pandas is a Python library used for data manip...,python.txt,0.405
3,What does cosine similarity compare?,Cosine similarity compares questions with docu...,project.txt,0.184
4,What are the college attendance rules?,Students should follow college attendance rules.,college.txt,0.394
5,Who won the FIFA World Cup in 2034?,Information not found in provided document.,Not found,0.000


In [11]:
test_df.to_csv(
    OUTPUT_DIR / "test_results.csv",
    index=False
)

with open(
    OUTPUT_DIR / "sample_transcripts.txt",
    "w",
    encoding="utf-8"
) as file:
    for question in test_questions:
        result = answer_question(question)

        file.write(f"Question: {question}\n")
        file.write(f"Answer: {result['answer']}\n")
        file.write(
            f"Source: {result.get('source') or 'Not found'}\n"
        )

        if result.get("passage"):
            file.write(
                f"Supporting passage: {result['passage']}\n"
            )

        file.write("\n" + "-" * 50 + "\n\n")

print("Test results and transcripts saved successfully!")

Test results and transcripts saved successfully!


In [12]:
print("Project folder:", PROJECT_DIR)
print("\nSaved output files:")

for file in sorted(OUTPUT_DIR.iterdir()):
    print(file.name)

Project folder: C:\Users\maham

Saved output files:
sample_transcripts.txt
test_results.csv


In [19]:
question = "What is machine learning?"

query_vector = vectorizer.transform([question])

scores = cosine_similarity(
    query_vector, tfidf_matrix
).flatten()

print("Question:", question)
print("Scores:", scores)
print("Best score:", scores.max())

Question: What is machine learning?
Scores: [0.04654982 0.24154318 0.         0.32482503]
Best score: 0.324825032002327


In [20]:
display_answer("What is machine learning?")

Question: What is machine learning?

Answer: Scikit-learn provides tools for machine learning.
Similarity score: 0.325

Source document: python.txt
Supporting passage: Python is a high-level programming language. Python is used in automation, web development,
data analysis, artificial intelligence and machine learning. Pandas is a Python library used for data manipulation. Scikit-learn provides tools for machine learning.

Top retrieved passages:


,source,chunk_id,passage,score
0,python.txt,1,Python is a high-level programming language. P...,0.325
1,machine_learning.txt,1,Machine learning is a branch of artificial int...,0.242
2,college.txt,1,Students should carry their college identifica...,0.047


{'answer': 'Scikit-learn provides tools for machine learning.',
 'source': 'python.txt',
 'passage': 'Python is a high-level programming language. Python is used in automation, web development,\ndata analysis, artificial intelligence and machine learning. Pandas is a Python library used for data manipulation. Scikit-learn provides tools for machine learning.',
 'score': 0.324825032002327,
 'retrieved':                  source  chunk_id  \
 0            python.txt         1   
 1  machine_learning.txt         1   
 2           college.txt         1   
 
                                              passage  score  
 0  Python is a high-level programming language. P...  0.325  
 1  Machine learning is a branch of artificial int...  0.242  
 2  Students should carry their college identifica...  0.047  }

In [21]:
def answer_question(question, top_k=3):
    question = question.strip()

    if not question:
        return {"answer": "Please enter a question."}

    query_vector = vectorizer.transform([question])
    chunk_texts = [chunk["text"] for chunk in chunks]

    chunk_scores = cosine_similarity(
        query_vector, tfidf_matrix
    ).flatten()

    ranked_indices = chunk_scores.argsort()[::-1][:top_k]

    retrieved = []
    for idx in ranked_indices:
        retrieved.append({
            "source": chunks[idx]["source"],
            "chunk_id": chunks[idx]["chunk_id"],
            "passage": chunks[idx]["text"],
            "score": round(float(chunk_scores[idx]), 3)
        })

    # Find the best matching sentence across all chunks
    candidates = []

    for idx, chunk in enumerate(chunks):
        sentences = re.split(
            r'(?<=[.!?])\s+', chunk["text"]
        )

        for sentence in sentences:
            if not sentence.strip():
                continue

            sentence_vector = vectorizer.transform([sentence])
            score = cosine_similarity(
                query_vector, sentence_vector
            )[0][0]

            candidates.append({
                "sentence": sentence,
                "source": chunk["source"],
                "passage": chunk["text"],
                "score": float(score)
            })

    best = max(candidates, key=lambda x: x["score"])

    if best["score"] < 0.12:
        return {
            "answer": "Information not found in provided document.",
            "source": None,
            "passage": None,
            "score": best["score"],
            "retrieved": pd.DataFrame(retrieved)
        }

    return {
        "answer": best["sentence"],
        "source": best["source"],
        "passage": best["passage"],
        "score": best["score"],
        "retrieved": pd.DataFrame(retrieved)
    }


print("Improved answer selection is ready!")


Improved answer selection is ready!


In [22]:
display_answer("What is machine learning?")

Question: What is machine learning?

Answer: Machine learning is a branch of artificial intelligence.
Similarity score: 0.503

Source document: machine_learning.txt
Supporting passage: Machine learning is a branch of artificial intelligence. It enables computers to learn patterns from data. Supervised learning uses labelled training examples. Classification predicts categories. Regression predicts numerical values.

Top retrieved passages:


,source,chunk_id,passage,score
0,python.txt,1,Python is a high-level programming language. P...,0.325
1,machine_learning.txt,1,Machine learning is a branch of artificial int...,0.242
2,college.txt,1,Students should carry their college identifica...,0.047


{'answer': 'Machine learning is a branch of artificial intelligence.',
 'source': 'machine_learning.txt',
 'passage': 'Machine learning is a branch of artificial intelligence. It enables computers to learn patterns from data. Supervised learning uses labelled training examples. Classification predicts categories. Regression predicts numerical values.',
 'score': 0.5033275339842773,
 'retrieved':                  source  chunk_id  \
 0            python.txt         1   
 1  machine_learning.txt         1   
 2           college.txt         1   
 
                                              passage  score  
 0  Python is a high-level programming language. P...  0.325  
 1  Machine learning is a branch of artificial int...  0.242  
 2  Students should carry their college identifica...  0.047  }

In [23]:
display_answer("Which library is used for data manipulation?")

Question: Which library is used for data manipulation?

Answer: Pandas is a Python library used for data manipulation.
Similarity score: 0.79

Source document: python.txt
Supporting passage: Python is a high-level programming language. Python is used in automation, web development,
data analysis, artificial intelligence and machine learning. Pandas is a Python library used for data manipulation. Scikit-learn provides tools for machine learning.

Top retrieved passages:


,source,chunk_id,passage,score
0,python.txt,1,Python is a high-level programming language. P...,0.405
1,machine_learning.txt,1,Machine learning is a branch of artificial int...,0.036
2,project.txt,1,A document question-answering assistant answer...,0.000


{'answer': 'Pandas is a Python library used for data manipulation.',
 'source': 'python.txt',
 'passage': 'Python is a high-level programming language. Python is used in automation, web development,\ndata analysis, artificial intelligence and machine learning. Pandas is a Python library used for data manipulation. Scikit-learn provides tools for machine learning.',
 'score': 0.7895623108901775,
 'retrieved':                  source  chunk_id  \
 0            python.txt         1   
 1  machine_learning.txt         1   
 2           project.txt         1   
 
                                              passage  score  
 0  Python is a high-level programming language. P...  0.405  
 1  Machine learning is a branch of artificial int...  0.036  
 2  A document question-answering assistant answer...  0.000  }

In [24]:
display_answer("Who won the FIFA World Cup in 2034?")

Question: Who won the FIFA World Cup in 2034?

Answer: Information not found in provided document.
Similarity score: 0.0

No sufficiently relevant source passage found.

Top retrieved passages:


,source,chunk_id,passage,score
0,python.txt,1,Python is a high-level programming language. P...,0.0
1,project.txt,1,A document question-answering assistant answer...,0.0
2,machine_learning.txt,1,Machine learning is a branch of artificial int...,0.0


{'answer': 'Information not found in provided document.',
 'source': None,
 'passage': None,
 'score': 0.0,
 'retrieved':                  source  chunk_id  \
 0            python.txt         1   
 1           project.txt         1   
 2  machine_learning.txt         1   
 
                                              passage  score  
 0  Python is a high-level programming language. P...    0.0  
 1  A document question-answering assistant answer...    0.0  
 2  Machine learning is a branch of artificial int...    0.0  }

In [25]:
display_answer(
    "Which library is used for data manipulation?"
)

Question: Which library is used for data manipulation?

Answer: Pandas is a Python library used for data manipulation.
Similarity score: 0.79

Source document: python.txt
Supporting passage: Python is a high-level programming language. Python is used in automation, web development,
data analysis, artificial intelligence and machine learning. Pandas is a Python library used for data manipulation. Scikit-learn provides tools for machine learning.

Top retrieved passages:


,source,chunk_id,passage,score
0,python.txt,1,Python is a high-level programming language. P...,0.405
1,machine_learning.txt,1,Machine learning is a branch of artificial int...,0.036
2,project.txt,1,A document question-answering assistant answer...,0.000


{'answer': 'Pandas is a Python library used for data manipulation.',
 'source': 'python.txt',
 'passage': 'Python is a high-level programming language. Python is used in automation, web development,\ndata analysis, artificial intelligence and machine learning. Pandas is a Python library used for data manipulation. Scikit-learn provides tools for machine learning.',
 'score': 0.7895623108901775,
 'retrieved':                  source  chunk_id  \
 0            python.txt         1   
 1  machine_learning.txt         1   
 2           project.txt         1   
 
                                              passage  score  
 0  Python is a high-level programming language. P...  0.405  
 1  Machine learning is a branch of artificial int...  0.036  
 2  A document question-answering assistant answer...  0.000  }

In [26]:
test_questions = [
    "What is Python used for?",
    "What is machine learning?",
    "Which library is used for data manipulation?",
    "What does cosine similarity compare?",
    "What are the college attendance rules?",
    "Who won the FIFA World Cup in 2034?",
    "What is the capital of Japan?",
    "What is quantum computing?"
]

test_results = []

for question in test_questions:
    result = answer_question(question)

    test_results.append({
        "Question": question,
        "Answer": result["answer"],
        "Source": result.get("source") or "Not found",
        "Similarity Score": round(result.get("score", 0), 3)
    })

test_df = pd.DataFrame(test_results)
display(test_df)

,Question,Answer,Source,Similarity Score
0,What is Python used for?,"Python is used in automation, web development,...",python.txt,0.407
1,What is machine learning?,Machine learning is a branch of artificial int...,machine_learning.txt,0.503
2,Which library is used for data manipulation?,Pandas is a Python library used for data manip...,python.txt,0.790
3,What does cosine similarity compare?,Cosine similarity compares questions with docu...,project.txt,0.522
4,What are the college attendance rules?,Students should follow college attendance rules.,college.txt,0.745
5,Who won the FIFA World Cup in 2034?,Information not found in provided document.,Not found,0.000
6,What is the capital of Japan?,Information not found in provided document.,Not found,0.000
7,What is quantum computing?,Information not found in provided document.,Not found,0.000


In [29]:
%pip install gradio pypdf

  Using cached typer-0.27.3-py3-none-any.whl.metadata (16 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached annotated_doc-0.0.5-py3-none-any.whl.metadata (6.5 kB)
  Using cached hf_xet-1.7.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/31.4 MB ? eta -:--:--
   - -------------------------------------- 1.3/31.4 MB 7.0 MB/s eta 0:00:05
   -- ------------------------------------- 2.1/31.4 MB 6.2 MB/s eta 0:00:05
   ---- ----------------------------------- 3.1/31.4 MB 5.2 MB/s eta 0:00:06
   ----- ---------------------------------- 4.2/31.4 MB 5.1 MB/s eta 0:00:06
   ------- -------------------------------- 5.5/31.4 MB 5.5 MB/s eta 0:00:05
   ---------

In [32]:
%pip install python-docx

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ------------- -------------------------- 1.3/4.0 MB 7.1 MB/s eta 0:00:01
   -------------------------- ------------- 2.6/4.0 MB 7.1 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 6.8 MB/s  0:00:00

   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [py

In [33]:
%pip install gradio pypdf python-docx scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [53]:
# ============================================================
# TASK 4: DOCUMENT QUESTION-ANSWERING ASSISTANT
# PDF / DOCX / TXT | TF-IDF | Supporting Evidence | Gradio
# ============================================================

import os
import re
import gradio as gr
import numpy as np

from docx import Document
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

FALLBACK = "Information not found in provided document"
document_index = None


# 1. EXTRACT DOCUMENT CONTENT
def extract_document(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    blocks = []

    if ext == ".txt":
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            blocks = [
                line.strip() for line in f.readlines()
                if line.strip()
            ]

    elif ext == ".pdf":
        reader = PdfReader(file_path)

        for page_no, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            if text.strip():
                blocks.append(f"Page {page_no}: {text.strip()}")

    elif ext == ".docx":
        doc = Document(file_path)

        for paragraph in doc.paragraphs:
            text = re.sub(r"\s+", " ", paragraph.text).strip()
            if text:
                blocks.append(text)

        for table in doc.tables:
            for row in table.rows:
                cells = [
                    re.sub(r"\s+", " ", cell.text).strip()
                    for cell in row.cells
                ]

                row_text = " | ".join(cells)
                if row_text.strip():
                    blocks.append(row_text)

    else:
        raise ValueError("Upload a PDF, DOCX, or TXT file.")

    return blocks


# 2. BUILD PASSAGES WITHOUT LOSING DOCUMENT CONTENT
def create_passages(blocks, max_words=130, overlap=25):
    passages = []

    # Retain paragraphs as the source of truth.
    # Join short paragraphs together, but keep table rows intact.
    current = []
    current_words = 0

    def save_current():
        nonlocal current, current_words

        if current:
            text = " ".join(current).strip()
            if text:
                passages.append({
                    "heading": "Document",
                    "text": text,
                    "display": text
                })

        current = []
        current_words = 0

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        # Keep each table row as an independent passage.
        if "|" in block:
            save_current()
            passages.append({
                "heading": "Document table",
                "text": block,
                "display": block
            })
            continue

        words = block.split()

        if current and current_words + len(words) > max_words:
            save_current()

        current.append(block)
        current_words += len(words)

        if current_words >= max_words:
            save_current()

    save_current()

    # Split any exceptionally long passage into overlapping chunks.
    final_passages = []

    for p in passages:
        words = p["text"].split()

        if len(words) <= max_words:
            final_passages.append(p)
            continue

        step = max_words - overlap

        for start in range(0, len(words), step):
            chunk = " ".join(words[start:start + max_words]).strip()

            if chunk:
                final_passages.append({
                    "heading": p["heading"],
                    "text": chunk,
                    "display": chunk
                })

    # Remove exact duplicates.
    unique = []
    seen = set()

    for p in final_passages:
        key = p["text"].lower()

        if key not in seen:
            seen.add(key)
            unique.append(p)

    return unique


# 3. BUILD SEARCH INDEX
def build_index(file_path):
    blocks = extract_document(file_path)

    if not blocks:
        raise ValueError("No readable text was found in the document.")

    passages = create_passages(blocks)

    if not passages:
        raise ValueError("No searchable passages were created.")

    corpus = [p["display"] for p in passages]

    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        sublinear_tf=True,
        stop_words="english"
    )

    matrix = vectorizer.fit_transform(corpus)

    return {
        "filename": os.path.basename(file_path),
        "blocks": blocks,
        "passages": passages,
        "vectorizer": vectorizer,
        "matrix": matrix
    }


# 4. LOOK UP HYPERPARAMETERS FROM THE DOCUMENT TABLE
def lookup_parameter(question, index):
    q = question.lower()

    parameters = [
        "n_estimators",
        "max_depth",
        "min_samples_split"
    ]

    requested = next(
        (p for p in parameters if p in q),
        None
    )

    if requested is None:
        return None

    for block in index["blocks"]:
        if "|" not in block:
            continue

        cells = [c.strip() for c in block.split("|")]

        if len(cells) >= 3 and cells[0].lower() == requested:
            return (
                f"The best value of {requested} is {cells[-1]}.",
                f"Source document: {index['filename']}\n"
                f"Supporting passage:\n{block}"
            )

    return None


# 5. RETRIEVE RELEVANT PASSAGES
def retrieve_passages(question, index, top_k=5):
    query_vector = index["vectorizer"].transform([question])

    scores = cosine_similarity(
        query_vector,
        index["matrix"]
    ).ravel()

    ranked = np.argsort(scores)[::-1]
    results = []

    for i in ranked[:top_k]:
        score = float(scores[i])

        results.append({
            "score": score,
            "heading": index["passages"][int(i)]["heading"],
            "text": index["passages"][int(i)]["text"]
        })

    return results


# 6. ANSWER QUESTIONS
def answer_question(question, index):
    q = question.lower().strip()

    # First handle parameter table questions.
    special = lookup_parameter(question, index)

    if special is not None:
        return special

    # Retrieve relevant passages.
    results = retrieve_passages(question, index, top_k=5)

    if not results:
        return (
            FALLBACK,
            f"Source document: {index['filename']}\n"
            "Supporting passage: None found."
        )

    # Feature questions: search passages for relevant project terminology.
    if any(term in q for term in [
        "feature", "features", "capability", "capabilities",
        "main function", "main functions"
    ]):
        feature_terms = [
            "phishing", "legitimate", "suspicious", "url",
            "classification", "risk", "probability",
            "dashboard", "security", "recommendation",
            "detection", "prediction", "user"
        ]

        matching = []

        for p in index["passages"]:
            text = p["text"].lower()
            hits = sum(term in text for term in feature_terms)

            if hits >= 2:
                matching.append((hits, p))

        if matching:
            matching.sort(key=lambda item: item[0], reverse=True)
            selected = [p for _, p in matching[:3]]

            answer = "\n\n".join(
                f"[Passage {i}]\n{p['text']}"
                for i, p in enumerate(selected, start=1)
            )

            sources = "\n\n".join(
                f"Source document: {index['filename']}\n"
                f"Supporting passage:\n{p['text']}"
                for p in selected
            )

            return answer, sources

    # General retrieval: do not reject all answers just because one token differs.
    best_score = results[0]["score"]

    if best_score < 0.12:
        return (
            FALLBACK,
            f"Source document: {index['filename']}\n"
            "Supporting passage: No sufficiently relevant passage found."
        )

    selected = [
        p for p in results
        if p["score"] >= max(0.03, best_score * 0.50)
    ][:3]

    answer = "\n\n".join(
        f"[Passage {i}]\n{p['text']}"
        for i, p in enumerate(selected, start=1)
    )

    sources = "\n\n".join(
        f"Source document: {index['filename']}\n"
        f"Similarity: {p['score']:.2%}\n"
        f"Supporting passage:\n{p['text']}"
        for p in selected
    )

    return answer, sources


# 7. LOAD DOCUMENT
def load_document(file_path):
    global document_index

    if file_path is None:
        document_index = None
        return "Please upload a document first."

    try:
        document_index = build_index(file_path)

        return (
            f"Document loaded: {document_index['filename']}\n"
            f"Passages created: {len(document_index['passages'])}\n"
            "You can now ask questions."
        )

    except Exception as error:
        document_index = None
        return f"Error processing document: {error}"


# 8. CHAT FUNCTION
def chat(message, history):
    global document_index

    history = history or []

    if not document_index:
        history += [
            {"role": "user", "content": message},
            {
                "role": "assistant",
                "content": "Please upload and load a document first."
            }
        ]
        return history, ""

    if not message or not message.strip():
        return history, ""

    answer, sources = answer_question(message, document_index)

    response = (
        f"{answer}\n\n"
        f"---\nEvidence and retrieval details:\n{sources}"
    )

    history += [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return history, ""


# 9. CREATE CHAT INTERFACE
with gr.Blocks(title="Document Question-Answering Assistant") as demo:
    gr.Markdown("# Document Question-Answering Assistant")
    gr.Markdown(
        "Upload a PDF, DOCX, or TXT document. "
        "Ask questions and inspect the supporting text passages."
    )

    file_input = gr.File(
        label="Upload Document",
        file_types=[".pdf", ".docx", ".txt"],
        type="filepath"
    )

    load_button = gr.Button("Load Document", variant="primary")
    status = gr.Textbox(label="Document Status", lines=3)

    chatbot = gr.Chatbot(label="Conversation", height=420)

    question_box = gr.Textbox(
        label="Ask a Question",
        placeholder="What is PhishGuard AI?"
    )

    with gr.Row():
        ask_button = gr.Button("Ask", variant="primary")
        clear_button = gr.Button("Clear Conversation")

    load_button.click(
        fn=load_document,
        inputs=[file_input],
        outputs=[status]
    )

    ask_button.click(
        fn=chat,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box]
    )

    question_box.submit(
        fn=chat,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box]
    )

    clear_button.click(
        fn=lambda: [],
        inputs=[],
        outputs=[chatbot]
    )


# 10. LAUNCH APPLICATION
demo.launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.
